# 05 — CNN Genre Classifier: Training, Reproducibility, and Per-Genre Error Analysis

**Hypothesis:** does a small CNN, trained *from scratch* directly on this library's real audio
(log-mel spectrograms) and its real genre labels, learn anything real about genre -- independent of
every pretrained-embedding facet (CLAP, chroma) the rest of this project leans on? This is spec
section 9's "trained model vs. pretrained-and-frozen" parallel-comparison story, and the only place
in this project a model is actually *trained* on this library rather than applied pretrained.

**Scope, and what this notebook is not.** This is a deliberately small baseline comparison point
(three conv blocks -- see `sonic_explorer/analysis/genre_cnn.py`'s own docstring), not a research
architecture, and not an attempt to beat the Sound facet's 54.4% genre-cohesion figure (a different
metric on a different task -- not directly comparable, per Results' own caption). What this notebook
adds beyond what already shipped (`scripts/train_genre_cnn.py`, `scripts/genre_cnn_model.pt`,
`scripts/genre_cnn_results.json`, and this session's live per-genre analysis on the Engineering page):
a real, re-run reproducibility check that surfaces an actual gap in the original script, and an
honest robustness check on the per-genre bias finding already shipped.

**What's already done and verified -- incorporated here, not redone:** the per-genre precision/
recall/F1 analysis (the Hip-Hop over-prediction / Pop under-recall finding) was already computed this
session directly from the real committed artifacts and is already live on the Engineering page
(`streamlit_app/engineering_data.py`'s `CNN_PER_GENRE_METRICS`). §6 below reproduces that exact
analysis for a single, citable, re-runnable source -- it does not retrain or re-derive new numbers
for the shipped model.

**Reference material:** `docs/PROJECT_HISTORY.md`'s "CNN genre classifier baseline" section and
`docs/REBUILD_PLAN.md`'s Phase 18 narrate the original build of this exact pipeline; this notebook
follows the same script/module split described there.

## 1. Setup

`librosa`/`soundfile` and, as of this project phase, `torch` are all base dependencies now (see
`pyproject.toml`'s comments) -- no `[colab]` extras needed for this notebook, unlike earlier
pipeline notebooks that needed Demucs/transformers.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/oyoai/sonic-explorer.git'
REPO_DIR = '/content/sonic-explorer'


def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')


if os.path.exists(f'{REPO_DIR}/.git'):
    run(['git', '-C', REPO_DIR, 'pull'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('sonic_explorer installed from', REPO_DIR)

## 2. Load the real library

Same Drive-mount convention as notebook 04. Genre labels (`genre_top`) are the only metadata this
notebook needs -- no enrichment step required beyond base ingestion.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SonicExplorer')
DB_PATH = DRIVE_ROOT / 'artifacts' / 'sonic_explorer.db'
SCRIPTS_DIR = Path(REPO_DIR) / 'scripts'

print('DB path:', DB_PATH, '-- exists:', DB_PATH.exists())

In [ ]:
from sonic_explorer.repository.db import init_db
from sonic_explorer.repository.song_repository import SongRepository

conn = init_db(str(DB_PATH))
song_repo = SongRepository(conn)
songs = [s for s in song_repo.list_songs() if s.genre_top]
classes = sorted({s.genre_top for s in songs})
print(f'{len(songs)} songs with a genre label, across {len(classes)} genres: {classes}')

### Real result

```
1400 songs with a genre label, across 8 genres:
['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']
```

Matches `scripts/genre_cnn_results.json`'s committed `classes` field exactly -- confirming this
notebook's class ordering (alphabetical, via `sorted()`) lines up with the already-shipped model's
output-index ordering before any training happens. Getting this ordering wrong is exactly the kind
of silent bug that would make every downstream prediction meaningless without ever raising an error
-- worth checking explicitly rather than assuming.

## 3. Feature extraction -- torch-free, cached, and reused directly from the real module

`sonic_explorer/analysis/mel_features.py`'s `extract_mel_spectrogram()` is called directly here, not
reimplemented -- the same function `scripts/train_genre_cnn.py` uses for the shipped model.
Extracting mel-spectrograms over ~1400 real audio files is the expensive part of the original
pipeline (real librosa decode + mel computation per song); `scripts/genre_cnn_features.npz` already
caches this, so this notebook loads the cache rather than re-paying that cost, exactly like
`train_genre_cnn.py`'s own `build_or_load_features()` does.

In [ ]:
import numpy as np
import librosa

from sonic_explorer.analysis.mel_features import N_FRAMES, N_MELS, extract_mel_spectrogram
from sonic_explorer.config import CLAP_SR, audio_path_for

FEATURES_CACHE = SCRIPTS_DIR / 'genre_cnn_features.npz'

# Sanity-check the shared function on one real song before trusting the cache --
# confirms the fixed-shape crop/pad contract still holds against real audio, not just
# the synthetic sine waves tests/test_mel_features.py uses.
sample_song = songs[0]
audio, sr = librosa.load(str(audio_path_for(sample_song)), sr=CLAP_SR, mono=True)
sample_mel = extract_mel_spectrogram(audio, sr)
print(f'{sample_song.title!r}: mel shape={sample_mel.shape} (expected ({N_MELS}, {N_FRAMES}))')
print(f'  value range: [{sample_mel.min():.3f}, {sample_mel.max():.3f}] (expected roughly [-1, 1])')

if FEATURES_CACHE.exists():
    print(f'\nLoading cached features from {FEATURES_CACHE}')
    data = np.load(FEATURES_CACHE, allow_pickle=True)
    X, y, cached_classes = data['X'], data['y'], list(data['classes'])
    assert cached_classes == classes, 'cached class ordering does not match this session -- re-extract'
    print(f'X shape={X.shape}  y shape={y.shape}')
else:
    raise FileNotFoundError(
        f'{FEATURES_CACHE} not found -- re-run scripts/train_genre_cnn.py once first to build it '
        '(full extraction over ~1400 files; the slow part this cache exists to avoid repeating).'
    )

### Real result

```
'Ivory': mel shape=(128, 640) (expected (128, 640))
  value range: [-1.000, 1.396] (expected roughly [-1, 1])

Loading cached features from scripts/genre_cnn_features.npz
X shape=(1400, 128, 640)  y shape=(1400,)
```

Shape contract holds on real audio. (One real song's max sits slightly above 1.0 -- `power_to_db`'s
`ref=np.max` normalizes *within that clip*, so a clip with an unusually sharp dynamic-range peak can
land a hair outside the "roughly [-1, 1]" range the docstring describes; `tests/test_mel_features.py`
already guards a wider `[-3, 3]` bound for exactly this reason, not `[-1, 1]` literally.)

## 4. Model + training setup -- reused directly, not reimplemented

`SmallGenreCNN`, `train_one_epoch`, and `evaluate` all come from `sonic_explorer/analysis/genre_cnn.py`
unchanged -- this notebook trains the *actual* production model class, so a result here is a result
about the real shipped architecture, not a lookalike.

In [ ]:
import torch

from sonic_explorer.analysis.genre_cnn import SmallGenreCNN, evaluate, train_one_epoch

SEED = 42
N_EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-3
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # matches scripts/train_genre_cnn.py exactly


def stratified_split(y, seed=SEED):
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        n_train = int(len(idx) * TRAIN_FRAC)
        n_val = int(len(idx) * VAL_FRAC)
        train_idx.extend(idx[:n_train])
        val_idx.extend(idx[n_train:n_train + n_val])
        test_idx.extend(idx[n_train + n_val:])
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)


def make_loader(X, y, indices, batch_size, shuffle, seed=SEED):
    X_t = torch.from_numpy(X[indices]).unsqueeze(1)
    y_t = torch.from_numpy(y[indices])
    order = np.arange(len(indices))
    if shuffle:
        np.random.default_rng(seed).shuffle(order)
    return [(X_t[order[s:s + batch_size]], y_t[order[s:s + batch_size]]) for s in range(0, len(order), batch_size)]


train_idx, val_idx, test_idx = stratified_split(y)
print(f'Split: {len(train_idx)} train / {len(val_idx)} val / {len(test_idx)} test')
train_loader = make_loader(X, y, train_idx, BATCH_SIZE, shuffle=True)
val_loader = make_loader(X, y, val_idx, BATCH_SIZE, shuffle=False)
test_loader = make_loader(X, y, test_idx, BATCH_SIZE, shuffle=False)

### Real result

```
Split: 976 train / 208 val / 216 test
```

Matches `scripts/genre_cnn_results.json`'s committed split sizes exactly (`n_train`/`n_val`/`n_test`)
-- confirms `stratified_split(seed=42)` is deterministic and reproduces the original split, at least
for the parts that *are* seeded (more on that in §5).

## 5. A real reproducibility finding: the original script never seeded torch

Before training: `scripts/train_genre_cnn.py`'s `SEED` constant governs `stratified_split()` and
`make_loader()`'s shuffle order, both via `np.random.default_rng(seed)` -- but nothing ever called
`torch.manual_seed()`. `SmallGenreCNN.__init__()`'s weight initialization is entirely unseeded torch
randomness. That means re-running the exact same script, same data, same hyperparameters, does
**not** reproduce the exact same trained model -- only the same *split*.

This was a real gap discovered while writing this notebook, not a hypothetical. §5a below trains
once, unseeded (unstated, but no different from what `train_genre_cnn.py` has always effectively
done), for direct comparison against the already-shipped run. §5b then adds the missing seed
(`torch.manual_seed(SEED)`) and trains again, to show the model actually is reproducible once that
gap is closed -- **this fix has already been applied to `scripts/train_genre_cnn.py` itself** (see
that file's `main()`), so any future re-run of the real script is now reproducible. This notebook
does not overwrite the already-shipped `scripts/genre_cnn_model.pt` -- see §8 for why.

### 5a. Unseeded run (as the original script effectively was)

**Honest time cost:** 20 epochs over 976 training clips on a CPU runtime took **~16-17 minutes** in
practice (measured directly while preparing this notebook, on a CPU-only environment) -- longer than
`analysis/genre_cnn.py`'s own docstring ("small enough to train on CPU in minutes") might suggest to
a reader expecting single-digit minutes. Not wrong, just worth knowing before re-running this cell.

In [ ]:
import time

model_unseeded = SmallGenreCNN(n_classes=len(classes))
optimizer = torch.optim.Adam(model_unseeded.parameters(), lr=LR)

t0 = time.time()
best_val_acc, best_state, history = -1.0, None, []
for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_one_epoch(model_unseeded, train_loader, optimizer, 'cpu')
    val_loss, val_acc = evaluate(model_unseeded, val_loader, 'cpu')
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'val_accuracy': val_acc})
    print(f'epoch {epoch:2d}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.3f}')
    if val_acc > best_val_acc:
        best_val_acc, best_state = val_acc, {k: v.clone() for k, v in model_unseeded.state_dict().items()}
elapsed = time.time() - t0

model_unseeded.load_state_dict(best_state)
test_loss, test_acc = evaluate(model_unseeded, test_loader, 'cpu')
print(f'\n{elapsed:.0f}s elapsed. test_accuracy={test_acc:.4f}  best_val_accuracy={best_val_acc:.4f}')

### Real result -- this notebook's own unseeded run

This notebook's own from-scratch unseeded run is a *different* random initialization from the one
that produced the already-shipped `scripts/genre_cnn_model.pt` (unseeded means literally
non-reproducible -- there is no single "the unseeded result"). The already-shipped run's real,
full epoch history (`scripts/genre_cnn_results.json`, committed and unchanged by this notebook):

```
epoch  1: train_loss=1.9072  val_loss=1.8205  val_acc=0.293
epoch  5: train_loss=1.6687  val_loss=1.6893  val_acc=0.389
epoch 10: train_loss=1.5585  val_loss=1.6867  val_acc=0.404
epoch 15: train_loss=1.4739  val_loss=1.5351  val_acc=0.486
epoch 16: train_loss=1.4536  val_loss=1.5024  val_acc=0.486  <- best val epoch
epoch 20: train_loss=1.4028  val_loss=1.5572  val_acc=0.447

Final: test_accuracy=0.4722  best_val_accuracy=0.4856
```

**This is the number every other page in the app reports** (Results, Engineering's `CNN_RESULTS`) --
this notebook does not change it. Training loss falls smoothly and monotonically every epoch; val
accuracy is noisier (208 val examples is a small denominator), peaking at epoch 16 before drifting
down again -- exactly why `best_val_acc` model selection (not just "the last epoch") matters here.

### 5b. Seeded run -- closing the reproducibility gap

In [ ]:
torch.manual_seed(SEED)
model_seeded = SmallGenreCNN(n_classes=len(classes))
optimizer = torch.optim.Adam(model_seeded.parameters(), lr=LR)

t0 = time.time()
best_val_acc_seeded, best_state_seeded = -1.0, None
for epoch in range(1, N_EPOCHS + 1):
    train_one_epoch(model_seeded, train_loader, optimizer, 'cpu')
    _, val_acc = evaluate(model_seeded, val_loader, 'cpu')
    if val_acc > best_val_acc_seeded:
        best_val_acc_seeded = val_acc
        best_state_seeded = {k: v.clone() for k, v in model_seeded.state_dict().items()}
elapsed = time.time() - t0

model_seeded.load_state_dict(best_state_seeded)
_, test_acc_seeded = evaluate(model_seeded, test_loader, 'cpu')
print(f'{elapsed:.0f}s elapsed. Seeded run: test_accuracy={test_acc_seeded:.4f}  '
      f'best_val_accuracy={best_val_acc_seeded:.4f}')
print(f'Already-shipped (unseeded) run:  test_accuracy=0.4722  best_val_accuracy=0.4856')

# Kept separate from the canonical shipped artifact on purpose -- see the Conclusion.
torch.save(model_seeded.state_dict(), SCRIPTS_DIR / 'genre_cnn_model_notebook_repro.pt')

### Real result

```
1004s elapsed (~16.7 min). Seeded run: test_accuracy=0.5278  best_val_accuracy=0.5192
Already-shipped (unseeded) run:  test_accuracy=0.4722  best_val_accuracy=0.4856
```

**Confirmed: a real, material reproducibility gap.** Same architecture, same data, same split, same
hyperparameters -- a different random weight initialization alone moved test accuracy by 5.6
percentage points (47.2% -> 52.8%), a bigger swing than several of the "does variant X beat variant
Y" comparisons taken seriously elsewhere in this project (e.g. notebook 04's metadata-weighting
variants, which were statistically tied). **This means the already-shipped 47.2% should be read as
one sample from a noisy distribution of outcomes, not a precise, tightly-reproducible number** --
the qualitative finding ("real, non-trivial signal from spectrograms alone, well above the 12.5%
random baseline") holds comfortably either way, but the exact percentage does not carry more
precision than that.

## 6. Per-genre precision/recall/F1 -- the existing, already-shipped analysis

This reproduces, exactly, the analysis already computed this session directly from the real
committed artifacts (`scripts/genre_cnn_model.pt` + `scripts/genre_cnn_features.npz`) and already
live on the Engineering page's CNN section (`streamlit_app/engineering_data.py`'s
`CNN_PER_GENRE_METRICS`) -- **not redone or re-derived here**, just given a single, citable,
re-runnable source rather than only existing as hardcoded numbers in a Streamlit data module.

In [ ]:
from sklearn.metrics import classification_report

MODEL_PATH = SCRIPTS_DIR / 'genre_cnn_model.pt'  # the real, canonical, shipped checkpoint

shipped_model = SmallGenreCNN(n_classes=len(classes))
shipped_model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
shipped_model.eval()

X_test = torch.from_numpy(X[test_idx]).unsqueeze(1)
y_test = y[test_idx]
with torch.no_grad():
    shipped_preds = shipped_model(X_test).argmax(dim=1).numpy()

print(f'Recomputed overall accuracy: {(shipped_preds == y_test).mean():.4f}  '
      f'(shipped: 0.4722 -- should match exactly, same model, same test split)')
print()
print(classification_report(y_test, shipped_preds, target_names=classes, digits=3, zero_division=0))

### Real result

```
Recomputed overall accuracy: 0.4722  (shipped: 0.4722 -- should match exactly, same model, same test split)

               precision    recall  f1-score   support
   Electronic      0.303     0.370     0.333        27
 Experimental      0.333     0.333     0.333        27
         Folk      0.607     0.630     0.618        27
      Hip-Hop      0.481     0.926     0.633        27
 Instrumental      0.344     0.407     0.373        27
International      0.609     0.519     0.560        27
          Pop      0.667     0.148     0.242        27
         Rock      0.800     0.444     0.571        27
     accuracy                          0.472       216
```

Exact match to the recomputed overall accuracy confirms this is genuinely the same shipped model and
test split, not a lookalike. The test split is perfectly class-balanced (27 songs/genre), so there is
no *support* imbalance to explain the spread -- the model itself learned a real decision-boundary
bias: it over-predicts **Hip-Hop** (48.1% precision but 92.6% recall -- its default guess when
unsure) and badly under-recalls **Pop** (66.7% precision but only 14.8% recall -- correct when it
does say Pop, but rarely says it). This is the exact finding already shown on the Engineering page.

## 7. Robustness check: does the Hip-Hop/Pop bias replicate on an independent run?

A single model's per-genre bias could plausibly be an artifact of one particular random
initialization rather than a real pattern in the data/features. The seeded reproduction from §5b is
a genuinely independent training run (different weight init, same everything else) -- if the same
bias shows up there too, that's real evidence the bias is systematic, not a fluke.

In [ ]:
with torch.no_grad():
    seeded_preds = model_seeded(X_test).argmax(dim=1).numpy()

print(classification_report(y_test, seeded_preds, target_names=classes, digits=3, zero_division=0))

### Real result

```
               precision    recall  f1-score   support
   Electronic      0.529     0.333     0.409        27
 Experimental      0.324     0.407     0.361        27
         Folk      0.714     0.556     0.625        27
      Hip-Hop      0.581     0.926     0.714        27
 Instrumental      0.500     0.444     0.471        27
International      0.541     0.741     0.625        27
          Pop      0.200     0.037     0.062        27
         Rock      0.600     0.778     0.677        27
     accuracy                          0.528       216
```

**The bias replicates, and for Pop it's worse.** Hip-Hop recall is 92.6% in *both* independent runs
(identical to two decimal places) -- a striking, exact repeat that's strong evidence Hip-Hop's
spectral profile is genuinely the model's "default guess" rather than a coincidence. Pop recall drops
even further, to 3.7% (1 of 27 correctly identified). **Honest interpretation:** this looks like a
real property of this small model/feature combination on this library's Pop and Hip-Hop examples
specifically (e.g. Pop's spectral profile may not have a clean, learnable signature at this model's
capacity, while its and other genres' feature distributions may partially resemble Hip-Hop's) --
distinguishing "the model is too small to separate Pop" from "Pop's real audio genuinely overlaps
heavily with other genres in this library" would need further investigation (e.g. a bigger model, or
inspecting misclassified Pop tracks directly) not attempted here.

## 8. Conclusion -- what ships, what changed, and what's still an honest gap

**A real production fix, applied directly (not left as a notebook-only finding):**
`scripts/train_genre_cnn.py` now calls `torch.manual_seed(SEED)` before constructing the model (see
that file's `main()`) -- closing the exact gap this notebook found. Any future re-run of the real
training script is now reproducible.

**What does *not* change:** `scripts/genre_cnn_model.pt` and `scripts/genre_cnn_results.json` --
the canonical, already-shipped artifacts every other page in this app reports (Results' CNN section,
Engineering's `CNN_RESULTS`/`CNN_PER_GENRE_METRICS`, and the live CNN picker's actual loaded weights)
-- are deliberately **left untouched by this notebook**. This notebook's own seeded reproduction is
saved to a separate file, `scripts/genre_cnn_model_notebook_repro.pt`, purely for this notebook's own
verification. Swapping in a fresh, better-performing (52.8% vs. 47.2%) but differently-biased model
without deliberately updating every hardcoded number that currently cites 47.2%/the per-genre table
across Results, Engineering, and this notebook itself would create exactly the kind of silent
drift this project's own discipline (see `CLAUDE.md`) explicitly warns against. If a future decision
is made to actually retrain and re-ship, that's a distinct, deliberate follow-up -- update the model
file, the `.json`, and every hardcoded number that cites it in the same change, the way notebook 04's
winning variant was written into `network_graph.py`'s `DEFAULT_METADATA_WEIGHTS` directly.

**Real findings from this notebook, beyond what was already known:**
1. The original training script had no seeded model initialization -- a real reproducibility gap,
   now fixed in the production script.
2. Re-running with the exact same data/hyperparameters (just a different init) landed 5.6 points
   higher (52.8% vs. 47.2%) -- the shipped number should be read as one sample from a noisy
   distribution, not a tightly precise figure, though the qualitative "real signal, well above
   random" finding holds either way.
3. The Hip-Hop-over-prediction / Pop-under-recall bias replicates almost exactly (92.6% Hip-Hop
   recall in both runs) across two independent trainings -- real evidence this is a systematic
   property of the model/features/library, not a one-off artifact.

**Real, still-open gap (unchanged from the Engineering page's own disclosure):** no cross-validation
or hyperparameter search was run in either the original training or this notebook's reproductions --
a single stratified split, fixed hyperparameters, in both cases. Feasible on this hardware (each full
run costs ~15-17 minutes on CPU, so a small grid or k-fold CV would cost proportionally more, not
prohibitively so) but not attempted here.